<a href="https://colab.research.google.com/github/skyBodrik/ColabResearch/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22lesson_06_pandas_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Занятие 6. Pandas: табличные данные и CSV

## Теория и практика программирования в Python
### Магистратура «Когнитивные исследования»

---

**Цели занятия:**
- Познакомиться с библиотекой pandas — главным инструментом для работы с табличными данными
- Научиться создавать, загружать, исследовать и фильтровать таблицы (DataFrame)
- Освоить чтение и запись CSV-файлов
- Научиться обрабатывать данные когнитивных экспериментов в табличном формате
- Практиковать промты к AI для задач на pandas

---
## Часть 1. Зачем pandas?

На предыдущих занятиях мы работали с данными в списках, словарях и массивах NumPy. Это удобно для однородных данных (только числа, только строки), но реальные данные экспериментов — это **таблицы**, где в разных столбцах хранятся разные типы:

| participant | condition | rt | correct | age |
|-------------|-----------|-----|---------|-----|
| 1 | congruent | 512 | True | 23 |
| 1 | incongruent | 743 | True | 23 |
| 2 | congruent | 487 | False | 25 |

Для таких данных нужен специальный инструмент — **pandas**.

### Что умеет pandas?

| Задача | Как делали раньше | Как с pandas |
|--------|-------------------|-------------|
| Загрузить CSV-файл | Читать строки, split, преобразовывать типы | `pd.read_csv('data.csv')` — одна строка! |
| Среднее по группам | Цикл, if, словарь | `df.groupby('condition')['rt'].mean()` |
| Отфильтровать выбросы | Цикл с проверками | `df[df['rt'] > 200]` |
| Добавить столбец | Новый список, цикл | `df['new'] = df['rt'] / 1000` |

**pandas** — самая важная библиотека для анализа данных в Python. В Google Colab она уже установлена.

In [ ]:
# Импортируем pandas — принятое сокращение pd
import pandas as pd
import numpy as np

print(f"pandas версия: {pd.__version__}")
print("pandas готов к работе!")

pandas версия: 2.2.2
pandas готов к работе!


---
## Часть 2. Series и DataFrame — два главных объекта pandas

В pandas есть два основных типа данных:

### Series — один столбец данных

**Series** — это одномерная структура, похожая на список или массив NumPy, но с **индексом** (подписями к каждому элементу).

Аналогия: Series — это **один столбец** таблицы.

In [ ]:
# Создаём Series из списка
rt = pd.Series([512, 487, 623, 701, 542])
print(rt)
print(f"\nТип: {type(rt)}")

0    512
1    487
2    623
3    701
4    542
dtype: int64

Тип: <class 'pandas.core.series.Series'>


In [ ]:
# Series с именованным индексом
accuracy = pd.Series(
    [0.94, 0.87, 0.91, 0.96, 0.82],
    index=['Алиса', 'Борис', 'Вера', 'Глеб', 'Дана'],
    name='accuracy'
)
print(accuracy)
print(f"\nТочность Алисы: {accuracy['Алиса']}")
print(f"Средняя точность: {accuracy.mean():.2f}")

Алиса    0.94
Борис    0.87
Вера     0.91
Глеб     0.96
Дана     0.82
Name: accuracy, dtype: float64

Точность Алисы: 0.94
Средняя точность: 0.90


### DataFrame — полноценная таблица

**DataFrame** — это двумерная структура: строки и столбцы. Каждый столбец — это Series.

Аналогия: DataFrame — это **лист Excel** или **таблица в SPSS**.

In [ ]:
# Создание DataFrame из словаря — самый частый способ
data = {
    'participant': [1, 1, 1, 2, 2, 2],
    'condition': ['congruent', 'incongruent', 'congruent',
                  'incongruent', 'congruent', 'incongruent'],
    'rt': [512, 743, 498, 678, 534, 701],
    'correct': [True, True, True, True, False, True]
}

df = pd.DataFrame(data)
df

,participant,condition,rt,correct
0,1,congruent,512,True
1,1,incongruent,743,True
2,1,congruent,498,True
3,2,incongruent,678,True
4,2,congruent,534,False
5,2,incongruent,701,True


In [ ]:
# Создание DataFrame из списка списков (каждый вложенный список — строка таблицы)
rows = [
    [1, 'congruent', 512, True],
    [1, 'incongruent', 743, True],
    [2, 'congruent', 487, False],
]

df2 = pd.DataFrame(rows, columns=['participant', 'condition', 'rt', 'correct'])
df2

,participant,condition,rt,correct
0,1,congruent,512,True
1,1,incongruent,743,True
2,2,congruent,487,False


### Чем DataFrame отличается от массива NumPy?

| Свойство | NumPy array | pandas DataFrame |
|----------|-------------|------------------|
| Размерность | Обычно одинаковый тип | Разные типы в столбцах |
| Столбцы | Нет имён (только номера) | Есть имена столбцов |
| Строки | Только номера | Есть индекс (номера или подписи) |
| Удобство | Быстрые вычисления | Удобная работа с таблицами |

---
## Часть 3. Чтение CSV-файлов: pd.read_csv()

**CSV** (Comma-Separated Values) — самый распространённый формат хранения экспериментальных данных. Это обычный текстовый файл, где значения разделены запятыми.

Пример содержимого CSV-файла:
```
participant,condition,rt,correct
1,congruent,512,True
1,incongruent,743,True
2,congruent,487,False
```

### Демонстрация: чтение CSV из строки

Чтобы попрактиковаться без внешних файлов, мы используем `io.StringIO` — это объект, который "притворяется" файлом. В реальной работе вы будете передавать путь к файлу или URL.

In [ ]:
import io

# CSV-данные в виде строки (имитация файла)
csv_text = """participant,condition,rt,correct,age
1,congruent,512,True,23
1,incongruent,743,True,23
1,congruent,498,True,23
2,incongruent,678,True,25
2,congruent,534,False,25
2,incongruent,701,True,25
3,congruent,456,True,22
3,incongruent,812,True,22
3,congruent,489,True,22
3,incongruent,756,False,22"""

# Читаем CSV с помощью pd.read_csv()
# io.StringIO превращает строку в "файл" — только для демонстрации!
df = pd.read_csv(io.StringIO(csv_text))
df

,participant,condition,rt,correct,age
0,1,congruent,512,True,23
1,1,incongruent,743,True,23
2,1,congruent,498,True,23
3,2,incongruent,678,True,25
4,2,congruent,534,False,25
5,2,incongruent,701,True,25
6,3,congruent,456,True,22
7,3,incongruent,812,True,22
8,3,congruent,489,True,22
9,3,incongruent,756,False,22


### Как это работает в реальной жизни

В Google Colab вы будете использовать `pd.read_csv()` с **путём к файлу** или **URL**:

```python
# Чтение из файла на компьютере или в Colab
df = pd.read_csv('experiment_data.csv')

# Чтение по URL (из интернета)
url = 'https://example.com/data.csv'
df = pd.read_csv(url)

# Чтение файла с разделителем-точкой с запятой (как в Excel на русской Windows)
df = pd.read_csv('data.csv', sep=';')

# Чтение с указанием кодировки
df = pd.read_csv('data.csv', encoding='utf-8')
```

**Частые параметры `pd.read_csv()`:**

| Параметр | Что делает | Пример |
|----------|-----------|--------|
| `sep` | Разделитель столбцов | `sep=';'`, `sep='\t'` (табуляция) |
| `encoding` | Кодировка файла | `encoding='utf-8'`, `encoding='cp1251'` |
| `header` | Номер строки с заголовками | `header=0` (по умолчанию), `header=None` |
| `index_col` | Столбец-индекс | `index_col=0` |
| `na_values` | Что считать пропуском | `na_values=['NA', 'N/A', '']` |

---
## Часть 4. Исследование данных

После загрузки данных первое, что нужно сделать, — **посмотреть**, что мы получили. pandas предоставляет множество удобных методов для этого.

Давайте создадим более реалистичный набор данных — результаты эксперимента Струпа:

In [ ]:
# Создаём данные эксперимента Струпа (30 проб)
np.random.seed(42)  # для воспроизводимости

n_trials = 30
participants = np.repeat([1, 2, 3, 4, 5], 6)  # 5 участников по 6 проб
conditions = ['congruent', 'incongruent'] * 15
ages = np.repeat([23, 25, 22, 27, 24], 6)

# RT зависит от условия: в неконгруэнтном условии медленнее
rt = []
for cond in conditions:
    if cond == 'congruent':
        rt.append(int(np.random.normal(500, 60)))
    else:
        rt.append(int(np.random.normal(700, 80)))

# Точность выше в конгруэнтном условии
correct = []
for cond in conditions:
    if cond == 'congruent':
        correct.append(np.random.random() > 0.05)  # 95% правильных
    else:
        correct.append(np.random.random() > 0.15)  # 85% правильных

stroop_df = pd.DataFrame({
    'participant': participants,
    'age': ages,
    'condition': conditions,
    'rt': rt,
    'correct': correct
})

print("Данные созданы!")
print(f"Размер таблицы: {stroop_df.shape[0]} строк, {stroop_df.shape[1]} столбцов")

Данные созданы!
Размер таблицы: 30 строк, 5 столбцов


In [ ]:
# .head() — первые 5 строк (можно указать другое число)
stroop_df.head()

,participant,age,condition,rt,correct
0,1,23,congruent,529,True
1,1,23,incongruent,688,True
2,1,23,congruent,538,True
3,1,23,incongruent,821,True
4,1,23,congruent,485,False


In [ ]:
# .tail() — последние 5 строк
stroop_df.tail(3)  # последние 3 строки

,participant,age,condition,rt,correct
27,5,24,incongruent,730,True
28,5,24,congruent,463,True
29,5,24,incongruent,676,True


In [ ]:
# .shape — размеры таблицы (строки, столбцы)
print(f"Размер: {stroop_df.shape}")
print(f"Строк: {stroop_df.shape[0]}")
print(f"Столбцов: {stroop_df.shape[1]}")

Размер: (30, 5)
Строк: 30
Столбцов: 5


In [ ]:
# .columns — имена столбцов
print("Столбцы:")
print(stroop_df.columns.tolist())

Столбцы:
['participant', 'age', 'condition', 'rt', 'correct']


In [ ]:
# .dtypes — типы данных в каждом столбце
print("Типы данных:")
print(stroop_df.dtypes)

Типы данных:
participant     int64
age             int64
condition      object
rt              int64
correct          bool
dtype: object


In [ ]:
# .info() — сводная информация о таблице
# Показывает типы, количество непустых значений, память
stroop_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   participant  30 non-null     int64 
 1   age          30 non-null     int64 
 2   condition    30 non-null     object
 3   rt           30 non-null     int64 
 4   correct      30 non-null     bool  
dtypes: bool(1), int64(3), object(1)
memory usage: 1.1+ KB


In [ ]:
# .describe() — описательная статистика для числовых столбцов
stroop_df.describe()

,participant,age,rt
count,30.00000,30.000000,30.000000
mean,3.00000,24.200000,586.133333
std,1.43839,1.749877,117.340569
min,1.00000,22.000000,396.000000
25%,2.00000,23.000000,475.250000
50%,3.00000,24.000000,586.500000
75%,4.00000,25.000000,681.000000
max,5.00000,27.000000,821.000000


**Что показывает `.describe()`:**

| Строка | Значение |
|--------|----------|
| `count` | Количество непустых значений |
| `mean` | Среднее арифметическое |
| `std` | Стандартное отклонение |
| `min` | Минимум |
| `25%` | 1-й квартиль (25-й перцентиль) |
| `50%` | Медиана (50-й перцентиль) |
| `75%` | 3-й квартиль (75-й перцентиль) |
| `max` | Максимум |

---
## Часть 5. Выбор столбцов

Чтобы работать с конкретными данными, нужно уметь выбирать столбцы.

In [ ]:
# Один столбец — получаем Series
rt_column = stroop_df['rt']
print(type(rt_column))
print(rt_column.head())

<class 'pandas.core.series.Series'>
0    529
1    688
2    538
3    821
4    485
Name: rt, dtype: int64


In [ ]:
# Со столбцом-Series можно делать вычисления
print(f"Среднее RT: {stroop_df['rt'].mean():.1f} мс")
print(f"Медиана RT: {stroop_df['rt'].median():.1f} мс")
print(f"Мин RT: {stroop_df['rt'].min()} мс")
print(f"Макс RT: {stroop_df['rt'].max()} мс")
print(f"Стд. отклонение: {stroop_df['rt'].std():.1f} мс")

Среднее RT: 586.1 мс
Медиана RT: 586.5 мс
Мин RT: 396 мс
Макс RT: 821 мс
Стд. отклонение: 117.3 мс


In [ ]:
# Несколько столбцов — передаём СПИСОК имён (двойные скобки!)
subset = stroop_df[['participant', 'condition', 'rt']]
print(type(subset))  # DataFrame, не Series!
subset.head()

<class 'pandas.core.frame.DataFrame'>


,participant,condition,rt
0,1,congruent,529
1,1,incongruent,688
2,1,congruent,538
3,1,incongruent,821
4,1,congruent,485


**Важно:** обратите внимание на двойные скобки `[[ ]]`!
- `df['rt']` — один столбец → **Series**
- `df[['rt']]` — список из одного столбца → **DataFrame**
- `df[['rt', 'correct']]` — два столбца → **DataFrame**

---
## Часть 6. Выбор строк: .loc[] и .iloc[]

pandas предоставляет два способа выбора строк:

| Метод | Тип обращения | Пример |
|-------|--------------|--------|
| `.iloc[]` | По **номеру** позиции (как в списке) | `df.iloc[0]` — первая строка |
| `.loc[]` | По **метке** индекса | `df.loc[0]` — строка с индексом 0 |

In [ ]:
# .iloc[] — выбор по номеру позиции (integer location)

# Первая строка
print("Первая строка:")
print(stroop_df.iloc[0])
print()

Первая строка:
participant            1
age                   23
condition      congruent
rt                   529
correct             True
Name: 0, dtype: object



In [ ]:
# Несколько строк (срез)
print("Строки 2-4:")
stroop_df.iloc[2:5]  # строки с индексами 2, 3, 4

Строки 2-4:


,participant,age,condition,rt,correct
2,1,23,congruent,538,True
3,1,23,incongruent,821,True
4,1,23,congruent,485,False


In [ ]:
# Конкретные строки и столбцы
# iloc[строки, столбцы]
print("Строки 0-2, столбцы 2-3 (rt и correct):")
stroop_df.iloc[0:3, 2:4]

Строки 0-2, столбцы 2-3 (rt и correct):


,condition,rt
0,congruent,529
1,incongruent,688
2,congruent,538


In [ ]:
# .loc[] — выбор по метке индекса и имени столбца

# Строка с индексом 0 (здесь совпадает с iloc, потому что индекс числовой)
print("Строка с индексом 0:")
print(stroop_df.loc[0])
print()

# Конкретное значение: строка 0, столбец 'rt'
print(f"RT первой пробы: {stroop_df.loc[0, 'rt']} мс")

Строка с индексом 0:
participant            1
age                   23
condition      congruent
rt                   529
correct             True
Name: 0, dtype: object

RT первой пробы: 529 мс


In [ ]:
# .loc с диапазоном строк и списком столбцов
stroop_df.loc[0:4, ['participant', 'condition', 'rt']]

,participant,condition,rt
0,1,congruent,529
1,1,incongruent,688
2,1,congruent,538
3,1,incongruent,821
4,1,congruent,485


**Когда разница между `.loc` и `.iloc` важна?**

Когда индекс — не стандартные числа 0, 1, 2... Например, после фильтрации строк индексы могут быть 0, 3, 7, 12... Тогда:
- `df.iloc[1]` — вторая строка таблицы (по позиции)
- `df.loc[1]` — строка с индексом 1 (которой может и не быть!)

---
## Часть 7. Фильтрация строк (булевая индексация)

Один из самых мощных инструментов pandas — **фильтрация по условию**. Это как задать вопрос: «Покажи мне только те строки, где...»

### Принцип работы

1. Создаём булеву маску (True/False для каждой строки)
2. Передаём маску в `df[...]`
3. Получаем только строки, где True

In [ ]:
# Шаг 1: создаём маску — логический вопрос к столбцу
mask = stroop_df['rt'] > 600
print("Маска (RT > 600):")
print(mask)
print(f"\nТип: {type(mask)}")

Маска (RT > 600):
0     False
1      True
2     False
3      True
4     False
5      True
6     False
7      True
8     False
9      True
10    False
11     True
12    False
13    False
14    False
15     True
16    False
17     True
18    False
19    False
20    False
21     True
22    False
23    False
24    False
25     True
26    False
27     True
28    False
29     True
Name: rt, dtype: bool

Тип: <class 'pandas.core.series.Series'>


In [ ]:
# Шаг 2: применяем маску — оставляем только строки, где True
slow_trials = stroop_df[stroop_df['rt'] > 600]
print(f"Проб с RT > 600 мс: {len(slow_trials)} из {len(stroop_df)}")
slow_trials

Проб с RT > 600 мс: 12 из 30


,participant,age,condition,rt,correct
1,1,23,incongruent,688,True
3,1,23,incongruent,821,True
5,1,23,incongruent,681,True
7,2,25,incongruent,761,True
9,2,25,incongruent,743,True
11,2,25,incongruent,662,True
15,3,22,incongruent,655,True
17,3,22,incongruent,725,True
21,4,27,incongruent,681,True
25,5,24,incongruent,708,True


In [ ]:
# Фильтрация по строковому столбцу
incongruent_trials = stroop_df[stroop_df['condition'] == 'incongruent']
print(f"Неконгруэнтных проб: {len(incongruent_trials)}")
incongruent_trials.head()

Неконгруэнтных проб: 15


,participant,age,condition,rt,correct
1,1,23,incongruent,688,True
3,1,23,incongruent,821,True
5,1,23,incongruent,681,True
7,2,25,incongruent,761,True
9,2,25,incongruent,743,True


In [ ]:
# Фильтрация по логическому столбцу
correct_trials = stroop_df[stroop_df['correct'] == True]
print(f"Правильных проб: {len(correct_trials)} из {len(stroop_df)}")

Правильных проб: 28 из 30


### Несколько условий: & (И) и | (ИЛИ)

**Важно!** При комбинировании условий:
1. Используйте `&` вместо `and` и `|` вместо `or`
2. **Каждое условие — в скобках!** Без скобок будет ошибка.

In [ ]:
# Два условия с & (И): неконгруэнтные И правильные
filtered = stroop_df[
    (stroop_df['condition'] == 'incongruent') & (stroop_df['correct'] == True)
]
print(f"Неконгруэнтные правильные пробы: {len(filtered)}")
filtered.head()

Неконгруэнтные правильные пробы: 15


,participant,age,condition,rt,correct
1,1,23,incongruent,688,True
3,1,23,incongruent,821,True
5,1,23,incongruent,681,True
7,2,25,incongruent,761,True
9,2,25,incongruent,743,True


In [ ]:
# Три условия: неконгруэнтные, правильные, RT от 200 до 1500
clean_data = stroop_df[
    (stroop_df['condition'] == 'incongruent') &
    (stroop_df['correct'] == True) &
    (stroop_df['rt'] > 200) &
    (stroop_df['rt'] < 1500)
]
print(f"Чистые неконгруэнтные пробы: {len(clean_data)}")
print(f"Среднее RT: {clean_data['rt'].mean():.1f} мс")

Чистые неконгруэнтные пробы: 15
Среднее RT: 683.3 мс


In [ ]:
# Условие с | (ИЛИ): участник 1 ИЛИ участник 3
subset = stroop_df[
    (stroop_df['participant'] == 1) | (stroop_df['participant'] == 3)
]
print(f"Проб участников 1 и 3: {len(subset)}")
subset.head()

Проб участников 1 и 3: 12


,participant,age,condition,rt,correct
0,1,23,congruent,529,True
1,1,23,incongruent,688,True
2,1,23,congruent,538,True
3,1,23,incongruent,821,True
4,1,23,congruent,485,False


In [ ]:
# Альтернатива: .isin() для проверки принадлежности к списку
subset = stroop_df[stroop_df['participant'].isin([1, 3, 5])]
print(f"Проб участников 1, 3, 5: {len(subset)}")

Проб участников 1, 3, 5: 18


### Типичная ошибка: забыть скобки

```python
# ЭТО ВЫЗОВЕТ ОШИБКУ!
df[df['condition'] == 'congruent' & df['correct'] == True]

# ПРАВИЛЬНО — каждое условие в скобках:
df[(df['condition'] == 'congruent') & (df['correct'] == True)]
```

Без скобок Python пытается сначала выполнить `'congruent' & df['correct']`, что не имеет смысла.

---
## Часть 8. Создание новых столбцов

Часто нужно добавить в таблицу вычисляемый столбец. Это делается одной строкой.

In [ ]:
# Создаём рабочую копию данных
df = stroop_df.copy()

# Новый столбец: RT в секундах
df['rt_sec'] = df['rt'] / 1000

# Новый столбец: логарифм RT (часто используется в психолингвистике)
df['log_rt'] = np.log(df['rt'])

# Новый столбец: отклонение от среднего RT
mean_rt = df['rt'].mean()
df['rt_deviation'] = df['rt'] - mean_rt

df[['participant', 'condition', 'rt', 'rt_sec', 'log_rt', 'rt_deviation']].head()

,participant,condition,rt,rt_sec,log_rt,rt_deviation
0,1,congruent,529,0.529,6.270988,-57.133333
1,1,incongruent,688,0.688,6.533789,101.866667
2,1,congruent,538,0.538,6.287859,-48.133333
3,1,incongruent,821,0.821,6.710523,234.866667
4,1,congruent,485,0.485,6.184149,-101.133333


In [ ]:
# Столбец на основе условия: быстрый или медленный ответ?
median_rt = df['rt'].median()
df['speed'] = np.where(df['rt'] <= median_rt, 'fast', 'slow')

df[['participant', 'condition', 'rt', 'speed']].head(10)

,participant,condition,rt,speed
0,1,congruent,529,fast
1,1,incongruent,688,slow
2,1,congruent,538,fast
3,1,incongruent,821,slow
4,1,congruent,485,fast
5,1,incongruent,681,slow
6,2,congruent,594,slow
7,2,incongruent,761,slow
8,2,congruent,471,fast
9,2,incongruent,743,slow


---
## Часть 9. Пропущенные значения (NaN)

В реальных данных часто встречаются пропуски — участник не ответил, датчик не сработал, данные потерялись. В pandas пропущенные значения обозначаются как `NaN` (Not a Number).

In [ ]:
# Создадим данные с пропусками
survey_data = pd.DataFrame({
    'participant': [1, 2, 3, 4, 5, 6],
    'age': [23, 25, np.nan, 27, 24, 22],
    'anxiety_score': [45, np.nan, 67, 32, np.nan, 78],
    'depression_score': [12, 15, np.nan, 8, 20, np.nan],
    'group': ['control', 'experimental', 'control', None, 'experimental', 'control']
})

survey_data

,participant,age,anxiety_score,depression_score,group
0,1,23.0,45.0,12.0,control
1,2,25.0,NaN,15.0,experimental
2,3,NaN,67.0,NaN,control
3,4,27.0,32.0,8.0,None
4,5,24.0,NaN,20.0,experimental
5,6,22.0,78.0,NaN,control


In [ ]:
# .isna() — показывает, где пропуски (True = пропуск)
print("Карта пропусков:")
print(survey_data.isna())
print()

# Сколько пропусков в каждом столбце?
print("Пропуски по столбцам:")
print(survey_data.isna().sum())

Карта пропусков:
   participant    age  anxiety_score  depression_score  group
0        False  False          False             False  False
1        False  False           True             False  False
2        False   True          False              True  False
3        False  False          False             False   True
4        False  False           True             False  False
5        False  False          False              True  False

Пропуски по столбцам:
participant         0
age                 1
anxiety_score       2
depression_score    2
group               1
dtype: int64


In [ ]:
# .dropna() — удаляет строки с любым пропуском
clean = survey_data.dropna()
print(f"До очистки: {len(survey_data)} строк")
print(f"После dropna(): {len(clean)} строк")
clean

До очистки: 6 строк
После dropna(): 1 строк


,participant,age,anxiety_score,depression_score,group
0,1,23.0,45.0,12.0,control


In [ ]:
# .dropna(subset=...) — удалить строки только если пропуск в определённых столбцах
# Удаляем только строки, где пропущен anxiety_score
clean2 = survey_data.dropna(subset=['anxiety_score'])
print(f"После dropna(subset=['anxiety_score']): {len(clean2)} строк")
clean2

После dropna(subset=['anxiety_score']): 4 строк


,participant,age,anxiety_score,depression_score,group
0,1,23.0,45.0,12.0,control
2,3,NaN,67.0,NaN,control
3,4,27.0,32.0,8.0,None
5,6,22.0,78.0,NaN,control


In [ ]:
# .fillna() — заполнить пропуски значением
# Часто пропуски заполняют средним или медианой
filled = survey_data.copy()

# Заполняем пропущенный возраст средним возрастом
filled['age'] = filled['age'].fillna(filled['age'].mean())

# Заполняем пропущенную группу значением 'unknown'
filled['group'] = filled['group'].fillna('unknown')

print("После заполнения пропусков:")
filled

После заполнения пропусков:


,participant,age,anxiety_score,depression_score,group
0,1,23.0,45.0,12.0,control
1,2,25.0,NaN,15.0,experimental
2,3,24.2,67.0,NaN,control
3,4,27.0,32.0,8.0,unknown
4,5,24.0,NaN,20.0,experimental
5,6,22.0,78.0,NaN,control


---
## Часть 10. Сортировка и подсчёт частот

### Сортировка: .sort_values()

In [ ]:
# Сортируем по RT (от меньшего к большему)
sorted_df = stroop_df.sort_values('rt')
sorted_df.head()

,participant,age,condition,rt,correct
14,3,22,congruent,396,True
26,5,24,congruent,430,True
16,3,22,congruent,439,True
18,4,27,congruent,445,True
28,5,24,congruent,463,True


In [ ]:
# Сортировка по убыванию
sorted_df = stroop_df.sort_values('rt', ascending=False)
sorted_df.head()

,participant,age,condition,rt,correct
3,1,23,incongruent,821,True
7,2,25,incongruent,761,True
9,2,25,incongruent,743,True
27,5,24,incongruent,730,True
17,3,22,incongruent,725,True


In [ ]:
# Сортировка по нескольким столбцам
# Сначала по участнику, потом по RT
sorted_df = stroop_df.sort_values(['participant', 'rt'])
sorted_df.head(10)

,participant,age,condition,rt,correct
4,1,23,congruent,485,False
0,1,23,congruent,529,True
2,1,23,congruent,538,True
5,1,23,incongruent,681,True
1,1,23,incongruent,688,True
3,1,23,incongruent,821,True
8,2,25,congruent,471,True
10,2,25,congruent,472,True
6,2,25,congruent,594,True
11,2,25,incongruent,662,True


### Подсчёт частот: .value_counts()

`.value_counts()` — незаменимый метод для категориальных данных. Показывает, сколько раз каждое значение встречается в столбце.

In [ ]:
# Сколько проб в каждом условии?
print("Частоты условий:")
print(stroop_df['condition'].value_counts())
print()

# Сколько проб у каждого участника?
print("Пробы по участникам:")
print(stroop_df['participant'].value_counts().sort_index())

Частоты условий:
condition
congruent      15
incongruent    15
Name: count, dtype: int64

Пробы по участникам:
participant
1    6
2    6
3    6
4    6
5    6
Name: count, dtype: int64


In [ ]:
# value_counts с нормализацией — показывает доли вместо абсолютных чисел
print("Доля правильных и неправильных ответов:")
print(stroop_df['correct'].value_counts(normalize=True))

Доля правильных и неправильных ответов:
correct
True     0.933333
False    0.066667
Name: proportion, dtype: float64


---
## Часть 11. Сохранение данных: .to_csv()

После обработки данных их нужно сохранить. Метод `.to_csv()` сохраняет DataFrame в CSV-файл.

In [ ]:
# Сохранение DataFrame в CSV
# В Colab файл сохранится в текущую папку (обычно /content/)

# Пример (раскомментируйте в Colab):
# stroop_df.to_csv('stroop_results.csv', index=False)
# print("Файл сохранён!")

# index=False — не сохранять индексы как отдельный столбец
# sep=';' — если нужен разделитель-точка с запятой
# encoding='utf-8-sig' — если будете открывать в Excel (русские буквы!)

# Демонстрация: превращаем DataFrame обратно в строку CSV
csv_output = stroop_df.head(3).to_csv(index=False)
print("Данные в формате CSV:")
print(csv_output)

Данные в формате CSV:
participant,age,condition,rt,correct
1,23,congruent,529,True
1,23,incongruent,688,True
1,23,congruent,538,True



---
## Часть 12. Практический пример: полный цикл обработки данных

Давайте пройдём весь путь от «сырых» данных до чистого набора для анализа.

**Задача:** у нас есть данные эксперимента Струпа. Нужно:
1. Загрузить данные
2. Посмотреть общую картину
3. Очистить от ошибочных проб
4. Удалить выбросы по RT
5. Добавить полезные столбцы
6. Сравнить условия

In [ ]:
# === Шаг 1: Загружаем данные ===

csv_stroop = """participant,condition,stimulus_word,ink_color,rt,response,correct,block
1,congruent,red,red,487,red,True,1
1,incongruent,red,blue,723,blue,True,1
1,congruent,green,green,512,green,True,1
1,incongruent,blue,green,891,red,False,1
1,congruent,blue,blue,45,blue,True,2
1,incongruent,green,red,678,red,True,2
1,congruent,red,red,501,red,True,2
1,incongruent,blue,red,1823,red,True,2
2,congruent,green,green,534,green,True,1
2,incongruent,red,green,701,green,True,1
2,congruent,blue,blue,489,blue,True,1
2,incongruent,green,blue,812,red,False,1
2,congruent,red,red,456,red,True,2
2,incongruent,blue,green,756,green,True,2
2,congruent,green,green,523,green,True,2
2,incongruent,red,blue,2150,blue,True,2
3,congruent,blue,blue,478,blue,True,1
3,incongruent,green,red,689,red,True,1
3,congruent,red,red,67,green,False,1
3,incongruent,blue,green,734,green,True,1
3,congruent,green,green,512,green,True,2
3,incongruent,red,blue,798,blue,True,2
3,congruent,blue,blue,498,blue,True,2
3,incongruent,green,red,645,red,True,2"""

raw_df = pd.read_csv(io.StringIO(csv_stroop))
print(f"Загружено {raw_df.shape[0]} строк, {raw_df.shape[1]} столбцов")
raw_df.head()

Загружено 24 строк, 8 столбцов


,participant,condition,stimulus_word,ink_color,rt,response,correct,block
0,1,congruent,red,red,487,red,True,1
1,1,incongruent,red,blue,723,blue,True,1
2,1,congruent,green,green,512,green,True,1
3,1,incongruent,blue,green,891,red,False,1
4,1,congruent,blue,blue,45,blue,True,2


In [ ]:
# === Шаг 2: Исследуем данные ===

print("=== Общая информация ===")
raw_df.info()

print("\n=== Описательная статистика ===")
print(raw_df.describe())

print("\n=== Распределение по условиям ===")
print(raw_df['condition'].value_counts())

print("\n=== Доля правильных ответов ===")
print(raw_df['correct'].value_counts(normalize=True))

=== Общая информация ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   participant    24 non-null     int64 
 1   condition      24 non-null     object
 2   stimulus_word  24 non-null     object
 3   ink_color      24 non-null     object
 4   rt             24 non-null     int64 
 5   response       24 non-null     object
 6   correct        24 non-null     bool  
 7   block          24 non-null     int64 
dtypes: bool(1), int64(3), object(4)
memory usage: 1.5+ KB

=== Описательная статистика ===
       participant           rt      block
count    24.000000    24.000000  24.000000
mean      2.000000   687.583333   1.500000
std       0.834058   450.382052   0.510754
min       1.000000    45.000000   1.000000
25%       1.000000   495.750000   1.000000
50%       2.000000   589.500000   1.500000
75%       3.000000   739.500000   2.000000
max    

In [ ]:
# === Шаг 3: Удаляем ошибочные пробы ===

n_before = len(raw_df)
df_correct = raw_df[raw_df['correct'] == True].copy()
n_after = len(df_correct)

print(f"Проб до фильтрации ошибок: {n_before}")
print(f"Проб после: {n_after}")
print(f"Удалено ошибочных: {n_before - n_after} ({(n_before - n_after)/n_before*100:.1f}%)")

Проб до фильтрации ошибок: 24
Проб после: 21
Удалено ошибочных: 3 (12.5%)


In [ ]:
# === Шаг 4: Удаляем выбросы по RT ===
# Стандартный критерий: RT < 200 мс или RT > 1500 мс

n_before = len(df_correct)
df_clean = df_correct[(df_correct['rt'] >= 200) & (df_correct['rt'] <= 1500)].copy()
n_after = len(df_clean)

print(f"Проб до фильтрации выбросов: {n_before}")
print(f"Проб после: {n_after}")
print(f"Удалено выбросов: {n_before - n_after}")

# Посмотрим, какие пробы были удалены
outliers = df_correct[(df_correct['rt'] < 200) | (df_correct['rt'] > 1500)]
print("\nУдалённые пробы:")
print(outliers[['participant', 'condition', 'rt']].to_string(index=False))

Проб до фильтрации выбросов: 21
Проб после: 18
Удалено выбросов: 3

Удалённые пробы:
 participant   condition   rt
           1   congruent   45
           1 incongruent 1823
           2 incongruent 2150


In [ ]:
# === Шаг 5: Добавляем полезные столбцы ===

# RT в секундах
df_clean['rt_sec'] = df_clean['rt'] / 1000

# Логарифм RT
df_clean['log_rt'] = np.log(df_clean['rt'])

# Совпадение слова и цвета (True/False)
df_clean['match'] = df_clean['stimulus_word'] == df_clean['ink_color']

df_clean[['participant', 'condition', 'stimulus_word', 'ink_color', 'rt', 'rt_sec', 'match']].head()

,participant,condition,stimulus_word,ink_color,rt,rt_sec,match
0,1,congruent,red,red,487,0.487,True
1,1,incongruent,red,blue,723,0.723,False
2,1,congruent,green,green,512,0.512,True
5,1,incongruent,green,red,678,0.678,False
6,1,congruent,red,red,501,0.501,True


In [ ]:
# === Шаг 6: Сравниваем условия ===

# Среднее RT по условиям
mean_congruent = df_clean[df_clean['condition'] == 'congruent']['rt'].mean()
mean_incongruent = df_clean[df_clean['condition'] == 'incongruent']['rt'].mean()
stroop_effect = mean_incongruent - mean_congruent

print("=== Результаты эксперимента Струпа ===")
print(f"Среднее RT (конгруэнтное):    {mean_congruent:.1f} мс")
print(f"Среднее RT (неконгруэнтное):  {mean_incongruent:.1f} мс")
print(f"Эффект Струпа:                {stroop_effect:.1f} мс")
print(f"Замедление:                   {stroop_effect/mean_congruent*100:.1f}%")

=== Результаты эксперимента Струпа ===
Среднее RT (конгруэнтное):    499.0 мс
Среднее RT (неконгруэнтное):  715.5 мс
Эффект Струпа:                216.5 мс
Замедление:                   43.4%


---
## Часть 13. Промты для работы с pandas

pandas — большая библиотека, и помощь AI здесь особенно полезна. Но важно правильно формулировать запросы.

### Пример 1: Фильтрация данных

---

#### Плохой промт:

> *"Отфильтруй данные в pandas"*

**Что не так:** AI не знает, какие данные, какие столбцы, по какому критерию фильтровать.

---

#### Хороший промт:

> *"У меня есть pandas DataFrame с данными эксперимента Струпа. Столбцы: participant (int), condition (str: 'congruent' или 'incongruent'), rt (int, время реакции в мс), correct (bool). Напиши код, который:*
> 1. *Оставляет только правильные ответы*
> 2. *Удаляет выбросы: RT < 200 или RT > 1500 мс*
> 3. *Выводит, сколько строк осталось после каждого этапа фильтрации*
> *DataFrame называется `df`. Используй pandas, без циклов."*

**Почему хорошо:**
- Указаны столбцы и их типы
- Чёткие критерии фильтрации
- Указано имя переменной
- Просьба без циклов — AI не будет писать `for` вместо векторных операций

### Пример 2: Создание нового столбца

---

#### Плохой промт:

> *"Добавь столбец"*

**Что не так:** Какой столбец? Как вычислять значения? В какой DataFrame?

---

#### Хороший промт:

> *"У меня есть DataFrame `df` со столбцом 'rt' (время реакции в миллисекундах, int). Мне нужно добавить новый столбец 'rt_zscore', содержащий z-нормированные значения RT. Формула z-score: (x - mean) / std. Используй pandas и numpy. Покажи пример результата с помощью .head()."*

**Почему хорошо:**
- Указаны входные данные
- Дана формула
- Указаны библиотеки
- Запрошена проверка результата

### Пример 3: Работа с пропусками

---

#### Плохой промт:

> *"Убери пропуски из данных"*

**Что не так:** Убрать строки целиком? Заполнить нулями? Средним? В каких столбцах?

---

#### Хороший промт:

> *"У меня DataFrame с результатами опросника. Столбцы: participant (int), age (float, есть пропуски), anxiety_score (float, есть пропуски), group (str, есть пропуски). Мне нужно:*
> 1. *Показать, сколько пропусков в каждом столбце*
> 2. *Заполнить пропущенный возраст медианой*
> 3. *Удалить строки, где пропущен anxiety_score (но НЕ удалять из-за пропусков в других столбцах)*
> 4. *Заполнить пропущенную группу строкой 'unknown'*
> *Используй pandas."*

**Почему хорошо:**
- Разные стратегии для разных столбцов — как в реальной работе
- AI не будет просто делать `dropna()` для всей таблицы

### Пример 4: Когда AI даёт устаревший или избыточный ответ

---

**Промт:** *"Посчитай среднее RT для каждого условия"*

**AI может выдать:**

In [ ]:
# Вариант 1 (от AI): с циклом — работает, но неоптимально
for condition in stroop_df['condition'].unique():
    subset = stroop_df[stroop_df['condition'] == condition]
    mean_rt = subset['rt'].mean()
    print(f"{condition}: {mean_rt:.1f} мс")

congruent: 488.9 мс
incongruent: 683.3 мс


In [ ]:
# Вариант 2: идиоматический pandas — одна строка!
print(stroop_df.groupby('condition')['rt'].mean())

condition
congruent      488.933333
incongruent    683.333333
Name: rt, dtype: float64


Оба варианта дают одинаковый результат, но второй — короче и быстрее. Если AI выдаёт цикл для задачи, которую pandas решает одной строкой, можно попросить:

> *"Перепиши без цикла, используя groupby. Это pandas-идиоматичный способ."*

Мы подробнее разберём `groupby` на следующих занятиях.

---
## Шпаргалка по pandas

### Создание и загрузка

```python
import pandas as pd

# Из словаря
df = pd.DataFrame({'col1': [...], 'col2': [...]})

# Из CSV
df = pd.read_csv('file.csv')
df = pd.read_csv('file.csv', sep=';', encoding='utf-8')
```

### Исследование

```python
df.head()        # первые 5 строк
df.tail()        # последние 5 строк
df.shape         # (строки, столбцы)
df.columns       # имена столбцов
df.dtypes        # типы данных
df.info()        # сводная информация
df.describe()    # статистика
```

### Выбор данных

```python
# Столбцы
df['col']              # один столбец (Series)
df[['col1', 'col2']]   # несколько столбцов (DataFrame)

# Строки
df.iloc[0]             # по номеру позиции
df.iloc[2:5]           # срез по позициям
df.loc[0, 'col']       # по индексу и имени столбца
```

### Фильтрация

```python
df[df['rt'] > 500]                                    # одно условие
df[(df['rt'] > 200) & (df['rt'] < 1500)]              # И (каждое в скобках!)
df[(df['cond'] == 'a') | (df['cond'] == 'b')]         # ИЛИ
df[df['participant'].isin([1, 2, 3])]                  # принадлежность списку
```

### Новые столбцы

```python
df['rt_sec'] = df['rt'] / 1000
df['log_rt'] = np.log(df['rt'])
df['speed'] = np.where(df['rt'] < 500, 'fast', 'slow')
```

### Пропущенные значения

```python
df.isna().sum()                # сколько пропусков
df.dropna()                    # удалить строки с пропусками
df.dropna(subset=['col'])      # удалить только если пропуск в col
df['col'].fillna(value)        # заполнить пропуски значением
```

### Сортировка и подсчёты

```python
df.sort_values('col')                    # по возрастанию
df.sort_values('col', ascending=False)   # по убыванию
df['col'].value_counts()                 # частоты значений
```

### Сохранение

```python
df.to_csv('output.csv', index=False)
```

---
## Практические задания

Выполните задания ниже прямо в этом ноутбуке. Для каждого задания напишите код в ячейке ниже вопроса.

### Задание 1. Создание DataFrame

Создайте DataFrame с результатами 6 участников теста Саймона (Simon task). В тесте Саймона стимул появляется справа или слева от центра экрана, и участник должен нажимать клавишу в зависимости от цвета стимула (не его расположения).

Столбцы:
- `participant`: номер участника (1-6)
- `side`: сторона стимула ('left' или 'right')
- `congruent`: совпадает ли сторона с клавишей ответа (True/False)
- `rt`: время реакции (придумайте реалистичные значения 350-800 мс)
- `correct`: правильность ответа (True/False)

Таблица должна содержать не менее 12 строк. Выведите первые 5 строк и описательную статистику.

In [ ]:
# Ваш код здесь


### Задание 2. Чтение CSV

Прочитайте следующие данные с помощью `pd.read_csv()` и `io.StringIO`. Это результаты опросника самооценки (шкала от 1 до 7).

```
id;gender;age;q1;q2;q3;q4;q5
1;F;21;5;6;4;5;6
2;M;23;3;4;2;3;4
3;F;22;6;7;5;6;7
4;M;25;2;3;1;2;3
5;F;20;4;5;3;4;5
6;M;24;5;4;6;5;4
7;F;22;7;6;7;7;6
8;M;23;1;2;1;1;2
```

Обратите внимание: разделитель — точка с запятой! Используйте параметр `sep=';'`.

Затем:
1. Выведите информацию о таблице (.info())
2. Покажите описательную статистику
3. Сколько мужчин и женщин в выборке?

In [ ]:
# Ваш код здесь


### Задание 3. Фильтрация данных

Используйте данные эксперимента Струпа (`stroop_df`, созданный в Части 4).

Выполните следующие фильтрации и для каждой выведите количество оставшихся строк:

1. Только пробы участника 3
2. Только конгруэнтные пробы с RT < 550 мс
3. Только неправильные ответы (correct == False)
4. Пробы с RT от 400 до 700 мс включительно
5. Пробы участников 1, 2 или 3 в конгруэнтном условии

In [ ]:
# Ваш код здесь


### Задание 4. Новые столбцы и обработка

Создайте DataFrame с данными визуального поиска (visual search task):

```python
search_data = pd.DataFrame({
    'participant': [1]*6 + [2]*6,
    'set_size': [4, 8, 16, 4, 8, 16] * 2,
    'target_present': [True, True, True, False, False, False] * 2,
    'rt': [420, 580, 890, 510, 750, 1200, 390, 540, 820, 480, 710, 1150]
})
```

Добавьте столбцы:
1. `rt_sec` — RT в секундах
2. `log_rt` — натуральный логарифм RT
3. `response_speed` — 'fast' если RT < 600, 'medium' если от 600 до 900, 'slow' если > 900

Подсказка для столбца 3: используйте `np.select()` или вложенный `np.where()`.

In [ ]:
# Ваш код здесь


### Задание 5. Пропущенные значения

Создайте и обработайте данные с пропусками:

```python
messy_data = pd.DataFrame({
    'participant': [1, 2, 3, 4, 5, 6, 7, 8],
    'age': [22, np.nan, 25, 21, 23, np.nan, 24, 22],
    'rt_mean': [520, 610, np.nan, 480, 550, 590, np.nan, 510],
    'accuracy': [0.92, 0.85, 0.91, np.nan, 0.88, 0.94, 0.87, 0.90],
    'handedness': ['right', 'left', 'right', 'right', None, 'right', 'left', 'right']
})
```

1. Покажите, сколько пропусков в каждом столбце
2. Заполните пропущенный возраст медианным возрастом
3. Удалите строки, где пропущено rt_mean
4. Заполните пропущенную рукость ('handedness') значением 'unknown'
5. Покажите итоговую таблицу и проверьте, что пропусков в обработанных столбцах не осталось

In [ ]:
# Ваш код здесь


### Задание 6. Сортировка и value_counts

Используйте данные эксперимента Струпа (`stroop_df`).

1. Отсортируйте данные по RT (от самого быстрого к самому медленному) и покажите 5 самых быстрых проб
2. Отсортируйте по участнику, затем по условию, затем по RT. Покажите результат
3. Используя `.value_counts()`, покажите:
   - Сколько проб каждого типа (condition)
   - Какую долю составляют правильные и неправильные ответы
   - Сколько проб у каждого участника в каждом условии (подсказка: можно передать список столбцов)

In [ ]:
# Ваш код здесь


### Задание 7. Полный цикл обработки данных

Перед вами «сырые» данные эксперимента на лексическое решение (lexical decision task). Участники видят строку букв и должны определить, слово это или нет.

```python
lex_csv = """participant,stimulus,word_type,rt,correct,trial_num
1,дом,word,450,True,1
1,кшп,nonword,520,True,2
1,мяч,word,35,True,3
1,влт,nonword,610,True,4
1,кот,word,480,True,5
1,рбг,nonword,1850,False,6
2,лес,word,510,True,1
2,нжм,nonword,580,True,2
2,рука,word,420,False,3
2,зщт,nonword,2200,True,4
2,стол,word,460,True,5
2,бкл,nonword,540,True,6
3,книга,word,490,True,1
3,вмщ,nonword,560,True,2
3,море,word,470,True,3
3,гтр,nonword,50,True,4
3,окно,word,500,True,5
3,шлв,nonword,610,False,6"""
```

Выполните полный цикл:
1. Загрузите данные
2. Исследуйте: .shape, .info(), .describe()
3. Удалите ошибочные пробы
4. Удалите выбросы (RT < 200 или RT > 1500)
5. Посчитайте, сколько данных потеряно на каждом шаге
6. Посчитайте среднее RT отдельно для слов (word) и неслов (nonword)
7. Какой тип стимулов распознаётся быстрее?

In [ ]:
# Ваш код здесь


### Задание 8. Работа с реальным форматом данных

В когнитивных экспериментах данные часто приходят в «неудобном» формате. Вот данные из PsychoPy (программа для создания экспериментов):

```python
psychopy_csv = """participant,trial,stim_text,stim_color,key_resp.keys,key_resp.rt,key_resp.corr
sub_01,1,RED,red,r,0.487,1
sub_01,2,GREEN,red,r,0.723,1
sub_01,3,BLUE,blue,b,0.512,1
sub_01,4,RED,green,g,0.891,1
sub_01,5,GREEN,green,g,0.456,1
sub_01,6,BLUE,red,g,0.678,0
sub_02,1,RED,red,r,0.534,1
sub_02,2,GREEN,blue,b,0.701,1
sub_02,3,BLUE,blue,b,0.489,1
sub_02,4,RED,green,r,0.812,0
sub_02,5,GREEN,green,g,0.523,1
sub_02,6,BLUE,red,r,0.756,1"""
```

1. Загрузите данные
2. Переименуйте столбцы для удобства: `key_resp.rt` -> `rt`, `key_resp.corr` -> `correct`
   (подсказка: `df.rename(columns={'старое': 'новое'})` или `df.columns = [...]`)
3. RT дано в секундах — создайте столбец `rt_ms` с RT в миллисекундах
4. Столбец correct содержит 0 и 1 — это нормально, pandas понимает 0 как False
5. Создайте столбец `condition`: 'congruent' если stim_text.lower() == stim_color, иначе 'incongruent'
6. Посчитайте среднее RT (в мс) и точность для каждого условия

In [ ]:
# Ваш код здесь


### Задание 9. Анализ опросника

Создайте DataFrame с ответами 10 студентов на 6 вопросов шкалы удовлетворённости обучением (от 1 до 5):

```python
survey = pd.DataFrame({
    'student_id': range(1, 11),
    'q1_content': [4, 5, 3, 4, 5, 2, 4, 5, 3, 4],
    'q2_delivery': [5, 4, 3, 5, 4, 3, 5, 4, 2, 4],
    'q3_materials': [3, 5, 4, 3, 5, 2, 4, 5, 3, 3],
    'q4_workload': [4, 3, 2, 4, 3, 1, 3, 4, 2, 3],
    'q5_support': [5, 4, 4, 5, 5, 3, 4, 5, 3, 4],
    'q6_overall': [4, 5, 3, 4, 5, 2, 4, 5, 3, 4]
})
```

1. Добавьте столбец `mean_score` — средний балл каждого студента по всем вопросам
2. Добавьте столбец `satisfaction` — 'high' если средний балл >= 4, 'medium' если 3-4, 'low' если < 3
3. Сколько студентов в каждой категории удовлетворённости?
4. Какой вопрос получил самые низкие оценки в среднем?
5. Отсортируйте студентов по среднему баллу (от высшего к низшему)

In [ ]:
# Ваш код здесь


### Задание 10 (с AI). Промт и критическая оценка

Придумайте задачу по обработке табличных данных (связанную с когнитивными экспериментами или опросниками).

1. **Напишите промт** для AI-ассистента (в текстовой ячейке ниже)
2. **Используйте AI** (Gemini, ChatGPT, Claude) — получите код
3. **Вставьте код** в ячейку кода
4. **Проверьте:** работает ли код? Верный ли результат?
5. **Оцените:** использует ли AI pandas идиоматично? Нет ли лишних циклов? Понятен ли код?
6. **Напишите** краткий разбор

**Мой промт:**

*(напишите здесь)*

In [ ]:
# Код от AI (вставьте сюда)


**Мой разбор:**

- Код работает: да/нет
- Результат верный: да/нет (как проверял/а)
- Использует ли AI pandas идиоматично (без лишних циклов):
- Что AI сделал хорошо:
- Что я бы изменил/а:
- Понимаю ли я каждую строку: да/нет (если нет — какие строки непонятны)

---
## Итоги занятия 6

Сегодня мы разобрали:

- Что такое pandas и зачем он нужен исследователю
- **Series** (один столбец) и **DataFrame** (таблица)
- Создание DataFrame из словарей и списков
- Чтение CSV: `pd.read_csv()` с разными параметрами
- Исследование данных: `.head()`, `.info()`, `.describe()`, `.shape`, `.columns`, `.dtypes`
- Выбор столбцов: `df['col']`, `df[['col1', 'col2']]`
- Выбор строк: `.loc[]`, `.iloc[]`
- Фильтрация по условиям: `&`, `|`, скобки обязательны!
- Создание новых столбцов
- Пропущенные значения: `.isna()`, `.dropna()`, `.fillna()`
- Сортировка `.sort_values()` и подсчёт частот `.value_counts()`
- Сохранение: `.to_csv()`
- Полный цикл обработки данных эксперимента

### Ключевой навык:
Умение загрузить, исследовать, очистить и подготовить данные — это **80% работы** при анализе экспериментов. Остальные 20% — статистика и визуализация.

### На следующем занятии:
Группировка данных (`groupby`), агрегация и сводные таблицы — как посчитать статистики отдельно для каждого участника, условия или группы.